# 02 — Batch RunYön grubu × seed kombinasyonlarını sırayla çalıştır.

In [ ]:
import os, sys

REPO_URL = 'https://github.com/hakanskn/CALMA.git'
REPO_DIR = '/content/arf_prototype'

if not os.path.exists(os.path.join(REPO_DIR, 'src')):
    os.system('rm -rf /content/CALMA')
    os.system(f'rm -rf {REPO_DIR}')
    ret = os.system(f'git clone {REPO_URL} /content/CALMA')
    if ret != 0:
        raise RuntimeError("git clone başarısız!")
    os.system(f'cp -r /content/CALMA/arf_prototype {REPO_DIR}')
    print('Repo hazır:', os.listdir(REPO_DIR))

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.environ.setdefault('DRIVE_BASE', '/content/drive/MyDrive/arf_results')

In [ ]:
from src.config import ExperimentConfig, DIRECTION_GROUPSfrom src.training.trainer import run_experimentimport time, jsondirection = 'A'              # 'A' | 'B' | 'C' | 'ALL'seeds = [42, 123]            # 1–5 seedepochs = 1                    # ilk test için 1limit_batches = 100           # tam set yerine smoke testmethods = DIRECTION_GROUPS[direction]print(f'{len(methods)} method × {len(seeds)} seed = {len(methods)*len(seeds)} run')

In [ ]:
results = []for m in methods:    for s in seeds:        cfg = ExperimentConfig.load(method=m, overrides={            'seed': s, 'num_epochs': epochs,            'limit_train_batches': limit_batches,            'limit_eval_batches': limit_batches // 2,            'run_id': f'run_{m}_s{s}_' + time.strftime('%Y%m%d_%H%M%S'),        })        print(f'▶ {m} seed={s}')        try:            r = run_experiment(cfg)            results.append((m, s, r['final_metrics']['test_ppl']))        except Exception as e:            print(f'  ✗ FAILED: {e}')            results.append((m, s, None))print('\nSummary:')for m, s, ppl in results:    print(f'  {m} seed={s}: PPL={ppl}')